# 02. 텍스트 분류 파인튜닝 — Trainer API

사전학습된 **DistilBERT** 를 IMDb 영화 리뷰 데이터로 **파인튜닝**해 긍정/부정 분류기를 만듭니다.
이것이 NLP에서 가장 표준적인 전이학습 방식이며, Hugging Face의 `Trainer` API로 학습 과정을 간결하게 구성합니다.

1. 데이터 로드 (IMDb)
2. 토크나이징
3. 모델 준비 (사전학습 + 분류 헤드)
4. Trainer로 학습 및 평가

In [ ]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import numpy as np
import evaluate
import torch

print('GPU:', torch.cuda.is_available())
checkpoint = 'distilbert-base-uncased'

## 1. 데이터 로드

IMDb 영화 리뷰 데이터셋을 불러옵니다. 학습 속도를 위해 일부만 추출합니다 (전체를 쓰려면 `.select` 줄을 지우세요).

In [ ]:
raw = load_dataset('stanfordnlp/imdb')
small_train = raw['train'].shuffle(seed=42).select(range(2000))
small_test = raw['test'].shuffle(seed=42).select(range(1000))
print('학습:', len(small_train), '| 테스트:', len(small_test))
print('예시:', small_train[0]['text'][:150], '...')

## 2. 토크나이징

텍스트를 모델이 이해하는 토큰 ID로 변환합니다. `map`으로 데이터셋 전체에 일괄 적용합니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

train_tok = small_train.map(tokenize_fn, batched=True)
test_tok = small_test.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 3. 모델 준비

사전학습 DistilBERT에 2개 클래스(긍정/부정) 분류 헤드를 얹습니다. 분류 헤드는 새로 초기화되어 파인튜닝됩니다.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

accuracy = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

## 4. 학습 및 평가

`TrainingArguments`로 하이퍼파라미터를 정하고, `Trainer`로 학습·평가를 수행합니다. GPU에서 수 분 내에 끝납니다.

In [ ]:
training_args = TrainingArguments(
    output_dir='/workspace/results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy='epoch',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
print('평가:', trainer.evaluate())

## 5. 추론

파인튜닝한 모델로 새 문장을 분류해 봅니다.

In [ ]:
from transformers import pipeline
clf = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)
print(clf('What a wonderful and touching film.'))
print(clf('This was a complete waste of time.'))

### 정리

- 사전학습 DistilBERT에 분류 헤드를 얹어 IMDb 데이터로 파인튜닝했습니다.
- `Trainer` API로 학습·평가·로깅을 간결하게 처리했습니다.
- 같은 패턴으로 감정분석 외 주제 분류, 스팸 탐지 등 다양한 텍스트 분류에 응용할 수 있습니다.